# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR<sup>2</sup>] dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the `mlcroissant` library.

### Dataset Source
The dataset is defined using the [Croissant schema](https://mlcommons.org/croissant/) and is loaded directly from a schema URL.

In [ ]:
# Install mlcroissant (if not already installed)
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and inspect basic information. All objects are referenced by their `@id` for clarity.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access and display metadata (as an object)
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

# If you want to review the full metadata fields:
# print(dataset.metadata)

## 2. Data Overview
Enumerate available record sets (`@id`), and for each record set, list its fields and columns by `@id` and name. This ensures all selections use canonical references.

In [ ]:
# This step inspects record sets and their details
print("Available record sets and their fields:")
record_sets = dataset.metadata.record_sets
for record_set in record_sets:
    print(f"\nRecordSet @id: {record_set.id}")
    print(f"  name    : {record_set.name}")
    print(f"  description: {record_set.description}")
    # List fields
    if hasattr(record_set, "fields") and record_set.fields:
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    # List columns (optional, not all record sets have columns)
    if hasattr(record_set, "columns") and record_set.columns:
        print("  Columns:")
        for column in record_set.columns:
            print(f"    - @id: {column.id} | name: {column.name}")

## 3. Data Extraction
Extract records by record set `@id`, load them as DataFrames, and display the first few records for preview. All selection is by `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Load each record set to a DataFrame (for demo, will do for first one only if there are large files)
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(3))
    except Exception as e:
        print(f"  Could not read records for {record_set_id}: {e}")

# Example: Preview one DataFrame
if dataframes:
    demo_record_set_id = list(dataframes.keys())[0]
    print(f"\nDemo DataFrame for RecordSet: {demo_record_set_id}")
    display(dataframes[demo_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
This section demonstrates filtering, normalization, and grouping using one numeric field.

**Note:** Please use the correct field `@id` for the specific analysis you want. The below example will attempt to select a numeric field automatically.

In [ ]:
import numpy as np

# Pick any dataframe and find a numeric field
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Try to pick numeric field from columns heuristically
    numeric_field_id = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break
    if numeric_field_id:
        print(f"Using numeric field @id: {numeric_field_id}")
        threshold = np.nanmean(df[numeric_field_id])
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try grouping by another field
        possible_groups = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        group_field = possible_groups[0] if possible_groups else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found in DataFrame.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Plot a histogram or scatterplot using fields (by `@id`).

In [ ]:
import matplotlib.pyplot as plt

# Visualization demo for one record set
if dataframes and numeric_field_id:
    plt.figure(figsize=(8, 5))
    df[numeric_field_id].dropna().hist(bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
else:
    print("No DataFrame or numeric field available for plot.")

## 6. Conclusion
- The dataset provides detailed records of mass spectrometry-based protein analysis from human mast cell extracellular vesicles, with richly structured Croissant metadata.
- Each entity is referenced by its canonical `@id`, supporting reproducible, standards-compliant data science workflows.
- Using `mlcroissant`, you can quickly load, filter, and analyze data by referring to record sets and fields using their `@id`s.
- Further analysis and domain-specific visualizations can be tailored using the data and metadata inspected above.